In [ ]:
import os

import general_utils as gu
import raster_utils as ru
import vector_utils as vu
import plot_utils as pu

In [ ]:

County="SB"
jsonBounds_file =r"C:\Users\Lauren\Desktop\sb_aoi.geojson"
project_crs="EPSG:32611"
procGrid_cellSize=20000
grid_file="C:/Users/Lauren/Desktop/"+County+"_grid.gpkg" ## output file 
STATE="CA"

In [ ]:
grid_gdf, bounds, bounds_poly = vu.create_proc_grid(bbox_json =jsonBounds_file, 
                        cell_size=procGrid_cellSize, 
                        out_epsg=project_crs, 
                        out_name=grid_file,
                        sampPts=True)

In [ ]:
import py3dep

dem = py3dep.static_3dep_dem(geometry=grid_gdf.dissolve().geometry.iloc[0], crs=grid_gdf.crs, resolution=10)
dem

In [ ]:
acs_var_id = "B01002_001E"
acs_yr = 2022
state_abr="CA"
region = "tract"

addr = "501 Poli St, Ventura, CA 93001"
subset_buff_dist_m = 60000


census_var = geou.get_census_var(region, state_abr, acs_var_id, acs_yr)

admin_bounds = geou.get_buffered_address_blocks(addr, subset_buff_dist_m)

census_merg = geou.merge_census(var=census_var, shapes=admin_bounds)

## static plot, no legend: ## merged_tracts.plot(column = var_id, cmap = "viridis", figsize = (8, 6))
## web map plot of variable id in viridis color map 
census_merg.explore(column = acs_var_id)

In [ ]:
BGs.plot()

In [ ]:
import pygris
from pygris import block_groups, roads


BGs = pygris.block_groups(state = "CA", county="Santa Barbara", cb = False)

roads = pygris.roads(state = "CA", county="Santa Barbara", cache = True)
##roads=gpd.read_file("SB_census.gpkg", layer="roads")

full_gdf_reproj=full_gdf.to_crs(BGs.crs)


BGs.to_file("SB_census.gpkg", driver="GPKG", layer="blockgroups")
roads.to_file("SB_census.gpkg", driver="GPKG",  layer="roads")
full_gdf_reproj.to_file("SB_census.gpkg", driver="GPKG", layer="soil")

In [ ]:
BGs.explode(index_parts=True).to_file("SB_census.gdb", driver="OpenFileGDB", layer="blockgroups") 
roads.to_file("SB_census.gdb", driver="OpenFileGDB", layer="roads") 
full_gdf_reproj.to_file("SB_census.gdb", driver="OpenFileGDB", layer="soil") 

In [ ]:
## list of files to merge as roads
files=gu.search_shp_subdir(in_dir, "shp")
## merge files
lines_gdf=vu.merge_vectors(files)
roads = lines_gdf.to_crs(4326)
#lines_gdf.explore()

In [11]:
pts = vu.lon_lat_to_gdf(points_df)
pts_tmp=pts[:-5]

In [ ]:
### calculate distance of line segment 

In [ ]:
## select points outside of parks 

## snap route to nearest primary road

import geopandas as gpd
from sqlalchemy import create_engine
from shapely.ops import nearest_points


#Join the points to the lines
points_join = gpd.sjoin_nearest(left_df=roads, right_df=pts_tmp, how="inner", max_distance=0.00005, distance_col="road_dist")
#Duplicate points/rows are created when there are multiple join features within max_distance. Drop the duplicates, keep the nearest
pts_tmp["pointgeom"] = pts_tmp.geometry #Save the point geometry or it is lost in spatial join
points_join = points_join.sort_values(by=["road_dist", "time"], ascending=True).drop_duplicates(subset="time", keep="first")
points_join["road_point"] = points_join.apply(lambda x: nearest_points(g1=x["geometry"], g2=x["pointgeom"])[0], axis=1) #Calculate nearest points on line and the joined point. Keep index [0], which is the point on the line nearest to the joined point


## Launch OSGeo4W Shell

> ogr2ogr -f "ESRI Shapefile" "C:\Users\Lauren\EO_ML\USCensus\SB_census_soil.shp" "C:\Users\Lauren
\EO_ML\USCensus\SB_census.gpkg" "soil"


In [ ]:
## microsoft access soil database 

import sqlite3
import os

in_db = r"C:\Users\Lauren\Downloads\wss_SSA_CA673_soildb_US_2003_[2023-09-12]\CA673\soildb_US_2003.mdb"

# Create a SQL connection to our SQLite database
con = sqlite3.connect(in_db)
cur = con.cursor()

# The result of a "cursor.execute" can be iterated over by row
# reading all table names
table_list = [a for a in cur.execute("SELECT name FROM sqlite_master WHERE type = 'table'")]
# here is you table list
print(table_list)


# Be sure to close the connection
con.close()

In [ ]:
in_dir=r"C:\Users\Lauren\Downloads\wss_SSA_CA672_soildb_US_2003_[2023-09-12]\CA672"

db_file=[i for i in os.listdir(in_dir) if i.endswith(".mdb")][0]
db_file

In [ ]:
sqlite3.connect('samp.mdb')
cursor = conn.execute("SELECT * from Sheet1")
for i in cursor:
    print(i)